In [2]:
import torch.nn as nn
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import torchvision.transforms as T
from utils import *
from torch.utils.data import DataLoader
from models import *
import random
from collections import Counter
from collections import OrderedDict
import seaborn as sns
import copy

cos = nn.CosineSimilarity(dim=0, eps=1e-9) 
device = "cuda"

# study 1 model's training with the new loss function with dist limits

# adjustable parameters
alpha_d = 100 # IID
iters = 12
n_clients = 100 # dataset size for one client
mali_local_ep = 5
global attack 
attack = "untargeted" #"backdoor", "tlp", "ut"
model_name = "ConvNet" # "resnet8", "ConvNet"
num_classes = 10
dataset ="fmnist" # "cifar10", "fmnist"

In [3]:
def filter_trainable_state_dict(model):
    """
    Filters model.state_dict() to retain only parameters that are in model.parameters().

    Args:
        model (torch.nn.Module): The model whose state_dict needs filtering.

    Returns:
        dict: Filtered state dictionary containing only trainable parameters.
    """
    param_names = {name for name, _ in model.named_parameters()}
    return {k: v for k, v in model.state_dict().items() if k in param_names}


In [4]:
def mali_model_craft_w_budget(model0, model1, model2, client_loader, budget):
    # craft model2 train with new loss function
    optimizer2 = optim.SGD(model2.parameters(), lr=0.05) # 0.001
    scheduler2 = torch.optim.lr_scheduler.StepLR(optimizer2, step_size=10, gamma=1)

    model2.load_state_dict(model1.state_dict())
    model2_result_before = eval_op_ensemble([model2], test_loader)
    
    train_rev_w_cos(model2, client_loader, optimizer2, scheduler2, epochs=4, 
                                    model0 = model0, 
                                    model1 = model1, 
                                    beta = 0.5, 
                                    budget = budget)
    
    cos_d = cos_dist(flat_dict(filter_trainable_state_dict(model1)) - flat_dict(filter_trainable_state_dict(model0)),
         flat_dict(filter_trainable_state_dict(model2)) - flat_dict(filter_trainable_state_dict(model0)))
    
    model2_result = eval_op_ensemble([model2], test_loader)
    
    return cos_d, model2_result, model2_result_before 

In [5]:
from collections import OrderedDict

def average_state_dicts(state_dicts):
    """
    Averages a list of PyTorch model state dictionaries.
    
    Args:
        state_dicts (list): A list of state_dicts from different models.
    
    Returns:
        OrderedDict: A state_dict with averaged parameters.
    """
    if not state_dicts:
        raise ValueError("The state_dicts list is empty.")
    
    # Initialize an empty OrderedDict to store the averaged values
    avg_state_dict = OrderedDict()
    
    # Get the keys from the first model (assumes all have the same keys)
    keys = state_dicts[0].keys()
    
    for key in keys:
        # Stack the tensors across models and compute the mean
        avg_state_dict[key] = sum(d[key] for d in state_dicts) / len(state_dicts)
    
    return avg_state_dict

In [6]:
def cos_dist(w1, w2):
    """Compute cosine similarity between two flattened weight tensors"""
    w1_flat, w2_flat = torch.cat([p.view(-1) for p in w1]), torch.cat([p.view(-1) for p in w2])
    return 1 - torch.dot(w1_flat, w2_flat) / (torch.norm(w1_flat) * torch.norm(w2_flat))

def get_delta_cos(model1, model2, model0_sd):
    flat_model0 = flat_dict(model0_sd)
    flat_model1 = flat_dict(model1.state_dict())
    flat_model2 = flat_dict(model2.state_dict())
    
    delta = torch.abs(flat_model1 - flat_model2)
    org_cos = cos((flat_model1 - flat_model0), (flat_model2 - flat_model0))
    return delta, 1-org_cos.item()

def model_eval(model, test_loader, attack):
    acc = eval_op_ensemble([model], test_loader)
    if attack == "tlp":
        asr = eval_op_ensemble_tr_lf_attack([model], test_loader)
    elif attack == "backdoor":
        asr = eval_op_ensemble_attack([model], test_loader)
    elif attack == "untargeted":
        asr = None
    return list(acc.values())[0], list(asr.values())[0]


def filter_trainable_state_dict(model):
    """
    Filters model.state_dict() to retain only parameters that are in model.parameters().

    Args:
        model (torch.nn.Module): The model whose state_dict needs filtering.

    Returns:
        dict: Filtered state dictionary containing only trainable parameters.
    """
    param_names = {name for name, _ in model.named_parameters()}
    return {k: v for k, v in model.state_dict().items() if k in param_names}


def train_rev_w_cos(model, loader, optimizer, scheduler, epochs, model0, model1, beta, budget):    
    model.train()
    # model.parameters need to use 
    flat_grad_model0 = flat_dict(filter_trainable_state_dict(model0))
    flat_grad_model1 = flat_dict(filter_trainable_state_dict(model1))
    grad_ben = (flat_grad_model1 - flat_grad_model0).to(device)
    
    losses = []
    running_loss, samples = 0.0, 0
    print(f"data length {len(loader) * loader.batch_size}: batches {len(loader)}, batch_size {loader.batch_size}")
    
    last_grad_mail = grad_ben
    for ep in range(epochs):
        for it, (x, y) in enumerate(loader):
            if it % 2 == 0:
                losses.append(round(eval_epoch(model, loader), 2))
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            
            # 1 negative CE loss
            loss_ce = nn.CrossEntropyLoss(reduction="mean")(model(x), y)
            loss_oppo_ce = - loss_ce

            running_loss += loss_oppo_ce.item() * y.shape[0]
            samples += y.shape[0]
            
            # add cos loss 
            w = torch.cat([p.view(-1) for p in model.parameters()]).to(device)
            grad_mail = w - flat_grad_model0
            target = torch.ones(len(w)).to(device)
            loss_cos = nn.CosineEmbeddingLoss()(grad_ben.unsqueeze(0), grad_mail.unsqueeze(0), target)
            
            # combindation loss
            loss_obj = (1-beta) * loss_oppo_ce + beta * loss_cos
            # only negative loss
            # loss_obj = loss_oppo_ce 
            
            loss_obj.backward()
            optimizer.step()
            scheduler.step()
            if it % 10 == 0:
                print(f"ep{ep}, loss_ce: {loss_oppo_ce:.0f}, loss_cos: {loss_cos:.4f}, loss_obj: {loss_obj:.0f}, lr: {optimizer.param_groups[0]['lr']}")
        
        # break
        crafted_cos_d = cos_dist_w(grad_ben, grad_mail)
        # print("eval losses", losses)
        print(f"cos_d: {crafted_cos_d}, budget: {budget}")

        if crafted_cos_d > budget:
            print(f"budget exceeded, finish training early, ep = {ep}")
            break
        
        last_grad_mail = grad_mail
        
    # craft_g = combine_tensors(B=grad_ben, M=grad_mail, budget=budget)
    craft_g = craft_tensor(B=grad_ben, M1=last_grad_mail, M2=grad_mail, k=budget)
    
    restored_crafted = restore_dict_grad_flat(craft_g, model0.state_dict(), model.state_dict())
    model.load_state_dict(restored_crafted)
    crafted_cos_d = cos_dist_w(grad_ben, craft_g)
        
    print(f"crafted cos_d: {crafted_cos_d}")        

    return {"loss": running_loss / samples}
    

In [7]:
def restore_dict_grad_dict(grad_dict, server_w, model_dict):
    state_dict_keys = set(model_dict.keys())
    param_dict_keys = set(server_w.keys())
    
    missing_keys = state_dict_keys - param_dict_keys    
    
    print("grad_dict", grad_dict)
    print("server_w", server_w)
    
    restored_w = {}

    for name, param in model_dict.items():
        if name not in missing_keys:
            print("name", name)
            print(grad_dict[name].shape, server_w[name].shape)

            restored_w[name] = grad_dict[name] + server_w[name]                           

        else:
            restored_w[name] = model_dict[name]
    return restored_w

In [44]:
# Define transformation (convert images to tensors and normalize)
transform_img = T.Compose([
    T.ToTensor(),  # Convert image to tensor
    T.Normalize((0.5,), (0.5,))  # Normalize the image with mean and std
])

if dataset == "fmnist":
    # Load the training dataset
    train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform_img)
    # Load the test dataset
    test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform_img)
elif dataset == "cifar10":
    train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_img)
    test_data = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_img)

# Create DataLoader for batch processing
client_loaders, test_loader, client_data_subsets =\
    data.get_loaders(train_data, test_data, n_clients,
                    alpha=alpha_d, batch_size=32, n_data=None, num_workers=4, seed=4)
    
model_fn = partial(models.get_model(model_name)[
                        0], num_classes=num_classes, dataset=dataset)

client_loader = client_loaders[0]

# created models 
model0 = model_fn().to(device) # orginal model
model1 = model_fn().to(device) # train with clean data
model2 = model_fn().to(device) # train with new loss function
model3 = model_fn().to(device)

model0_sd = {k: v.clone().detach() for k, v in model1.state_dict().items()}

optimizer0 = optim.SGD(model0.parameters(), lr=0.001)
optimizer1 = optim.SGD(model1.parameters(), lr=0.001)
optimizer2 = optim.SGD(model2.parameters(), lr=0.001)
optimizer3 = optim.SGD(model3.parameters(), lr=0.001)



Data split:
 - Client 0: [62 62 58 65 57 61 58 53 56 63]                         -> sum=595
 - Client 1: [61 64 63 55 59 58 60 65 57 58]                         -> sum=600
 - Client 2: [54 57 59 60 64 66 62 63 60 54]                         -> sum=599
 - Client 3: [62 56 63 56 62 63 59 58 61 61]                         -> sum=601
 - Client 4: [57 67 62 56 57 65 65 57 54 58]                         -> sum=598
 - Client 5: [54 51 65 59 66 67 53 57 62 68]                         -> sum=602
 - Client 6: [61 65 60 67 53 60 61 50 58 66]                         -> sum=601
 - Client 7: [53 58 49 70 75 63 51 61 54 64]                         -> sum=598
 - Client 8: [64 58 60 52 59 66 65 64 55 59]                         -> sum=602
 - Client 9: [58 63 46 68 61 56 57 66 69 55]                         -> sum=599
.  .  .  .  .  .  .  .  .  .  
.  .  .  .  .  .  .  .  .  .  
.  .  .  .  .  .  .  .  .  .  
 - Client 91: [54 61 56 53 60 63 63 62 70 58]                         -> sum=600
 - Client 92: 

In [32]:
epochs=5
# train model 1 with clean data to maturity
model1.load_state_dict(model0.state_dict())
for ep in range(epochs):
    last_model= copy.deepcopy(model1)
    train_op(model1, client_loader, optimizer1, epochs=1, print_train_loss=True)
    acc = eval_op_ensemble([model1], test_loader)
    print(f"ep {ep}, acc {acc}")
    

ben_update = flat_dict(filter_trainable_state_dict(model1)) - flat_dict(filter_trainable_state_dict(last_model))
ben_norm =torch.norm(ben_update, p=2)

[2.37, 2.31, 2.26, 2.21, 2.16, 2.12, 2.07, 2.03, 2.0, 1.97]
ep 0, acc {'test_accuracy': 0.5152}
[1.95, 1.92, 1.89, 1.86, 1.84, 1.81, 1.78, 1.76, 1.74, 1.71]
ep 1, acc {'test_accuracy': 0.6053}
[1.7, 1.68, 1.66, 1.64, 1.62, 1.6, 1.59, 1.57, 1.55, 1.54]
ep 2, acc {'test_accuracy': 0.6501}
[1.53, 1.52, 1.5, 1.49, 1.48, 1.46, 1.45, 1.44, 1.42, 1.41]
ep 3, acc {'test_accuracy': 0.669}
[1.41, 1.39, 1.38, 1.37, 1.36, 1.35, 1.34, 1.33, 1.32, 1.31]
ep 4, acc {'test_accuracy': 0.6955}


In [13]:
ben_norm

tensor(0.0443, device='cuda:0')

In [14]:
# first attack, reverse the direction of the gradient
mali_update = - ben_update
r_w = restore_dict_grad_flat(mali_update, last_model.state_dict(), last_model.state_dict())
model2.load_state_dict(r_w)


<All keys matched successfully>

In [18]:
model2_result = eval_op_ensemble([model2], test_loader)
model2_result, list(acc.values())[0]-list(model2_result.values())[0]

({'test_accuracy': 0.6504}, 0.043200000000000016)

In [45]:
# second attack, flip the class labels
model2.load_state_dict(last_model.state_dict())
loss = train_op_flip(model2, client_loader, optimizer2, epochs=10, class_num=10)
print("loss", loss)
model2_result = eval_op_ensemble([model2], test_loader)
print("model2_result",model2_result)


loss {'loss': 1.5351643287233945}
model2_result {'test_accuracy': 0.0223}


In [53]:
mali_update = flat_dict(filter_trainable_state_dict(model2)) - flat_dict(filter_trainable_state_dict(model1))
print("mali_update", torch.norm(mali_update, p=2) )
mali_update = mali_update / (torch.norm(mali_update, p=2) +1e-8) * ben_norm

print("mali_update", torch.norm(mali_update, p=2) )
r_w = restore_dict_grad_flat(mali_update, last_model.state_dict(), last_model.state_dict())
model2.load_state_dict(r_w)

model2_result = eval_op_ensemble([model2], test_loader)
model2_result, list(acc.values())[0]-list(model2_result.values())[0]

mali_update tensor(16.4823, device='cuda:0')
mali_update tensor(0.0437, device='cuda:0')


({'test_accuracy': 0.6689}, 0.026599999999999957)

In [ ]:
# third attack craft the gradient with the new loss function

In [49]:
# train model 2 from model 1 
cos_d, model2_result, model2_result_before = mali_model_craft_w_budget(model0=last_model,
                                                                        model1=model1, 
                                                                        model2=model2, 
                                                                        client_loader=client_loader, 
                                                                        budget=0.5)

data length 608: batches 19, batch_size 32
ep0, loss_ce: -2, loss_cos: 0.0000, loss_obj: -1, lr: 0.05
ep0, loss_ce: -163, loss_cos: 0.0529, loss_obj: -82, lr: 0.05
cos_d: 0.2631532549858093, budget: 0.5
ep1, loss_ce: -815, loss_cos: 0.2973, loss_obj: -407, lr: 0.05
ep1, loss_ce: -3902, loss_cos: 0.6418, loss_obj: -1951, lr: 0.05
cos_d: 0.8000919222831726, budget: 0.5
budget exceeded, finish training early, ep = 1
alpha_tensor tensor(0.8019)
crafted cos_d: 0.5


In [50]:
model2_result, cos_d

({'test_accuracy': 0.1}, tensor(0.5000, device='cuda:0'))

In [51]:
mali_update = flat_dict(filter_trainable_state_dict(model2)) - flat_dict(filter_trainable_state_dict(last_model))
mali_update = mali_update / (torch.norm(mali_update, p=2) +1e-8) * ben_norm

print("mali_update", torch.norm(mali_update, p=2) )
r_w = restore_dict_grad_flat(mali_update, last_model.state_dict(), last_model.state_dict())
model2.load_state_dict(r_w)

model2_result = eval_op_ensemble([model2], test_loader)
model2_result, list(acc.values())[0]-list(model2_result.values())[0]

mali_update tensor(0.0437, device='cuda:0')


({'test_accuracy': 0.6767}, 0.01880000000000004)